# Modelling
Run `01_data_processing.ipynb` first.

Trains and evaluates the models, then saves the predictions and best model for the last three graph notebooks.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

os.makedirs("figures", exist_ok=True)
RANDOM_STATE = 42

In [2]:
fe = pd.read_pickle("processed/features.pkl")
feature_cols = pd.read_json("processed/feature_cols.json")[0].tolist()

## 5. Model implementation
**Validation:** time-based split — train on everything before 1 Oct 2012, test on the last 3 months.
A random split would let the model see hours right next to the test hours (near-identical), giving over-optimistic scores.

In [3]:
split_date = pd.Timestamp("2012-10-01")
train, test = fe[fe["datetime"] < split_date], fe[fe["datetime"] >= split_date]
X_train, X_test = train[feature_cols], test[feature_cols]
y_train_log, y_test = train["log_cnt"], test["cnt"]
print(f"Train: {len(train)} hours ({train['datetime'].min():%d %b %Y} – {train['datetime'].max():%d %b %Y})")
print(f"Test:  {len(test)} hours ({test['datetime'].min():%d %b %Y} – {test['datetime'].max():%d %b %Y})")

Train: 15211 hours (01 Jan 2011 – 30 Sep 2012)
Test:  2168 hours (01 Oct 2012 – 31 Dec 2012)


In [4]:
def evaluate(name, preds):
    return {"Model": name,
            "RMSE": mean_squared_error(y_test, preds) ** 0.5,
            "MAE": mean_absolute_error(y_test, preds),
            "R2": r2_score(y_test, preds)}

results, predictions = [], {}

# Baseline: average rentals for that hour + working-day combination in the training data
baseline_lookup = train.groupby(["hr", "workingday"])["cnt"].mean()
predictions["Baseline (hourly average)"] = test.set_index(["hr", "workingday"]).index.map(baseline_lookup).values

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, random_state=RANDOM_STATE),
}
for name, model in models.items():
    model.fit(X_train, y_train_log)
    predictions[name] = np.clip(np.expm1(model.predict(X_test)), 0, None)   # back to bike counts

for name, preds in predictions.items():
    results.append(evaluate(name, preds))

results_df = pd.DataFrame(results).set_index("Model").round(3)
results_df

,RMSE,MAE,R2
Model,,,
Baseline (hourly average),119.589,79.950,0.648
Linear Regression,155.688,90.703,0.404
Random Forest,78.246,51.419,0.849
Gradient Boosting,67.016,42.815,0.889


## 6. Model evaluation
### 6.1 Cross-validation on the training period (walk-forward)
Checks the scores are stable, not just lucky on one test period.

In [5]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
tscv = TimeSeriesSplit(n_splits=5)
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train_log, cv=tscv, scoring="neg_root_mean_squared_error")
    print(f"{name:20s} RMSE on log scale: {-scores.mean():.3f} ± {scores.std():.3f}")

Linear Regression    RMSE on log scale: 0.752 ± 0.074


Random Forest        RMSE on log scale: 0.479 ± 0.134


Gradient Boosting    RMSE on log scale: 0.440 ± 0.146


## Results table

In [6]:
results_df.to_csv("figures/results_table.csv")
print(results_df.to_string())

                              RMSE     MAE     R2
Model                                            
Baseline (hourly average)  119.589  79.950  0.648
Linear Regression          155.688  90.703  0.404
Random Forest               78.246  51.419  0.849
Gradient Boosting           67.016  42.815  0.889


## Save for the graph notebooks

In [7]:
best_name = results_df.drop(index="Baseline (hourly average)")["RMSE"].idxmin()
joblib.dump({"best_name": best_name, "best_model": models[best_name], "predictions": predictions,
             "X_test": X_test, "y_test": y_test, "test": test, "feature_cols": feature_cols},
            "processed/model_results.joblib", compress=3)
print("Best model:", best_name)

Best model: Gradient Boosting
